# Two agents and a wall

A folder of customer records, and a billing fault nobody in the building knows the rule for. The local model is the only thing that reads a record or writes a fix. The hosted model gets the shape of the problem and sends back a plan.

In [ ]:
import re
from pathlib import Path
from shutil import copytree
from tempfile import mkdtemp

from fastcore.all import L
from ramabana.agent import Agent
from ramabana.core import resolve
from ramabana.tools import LocalHost
from ramabana.vault import VaultHost
from vishalakshi import Vault
from vishalakshi.pii import pii_report

LOCAL, CLOUD = 'gemma-e4b', 'sonnet'
DEVICE = '{0.backend}/{0.model_id}'.format(resolve(LOCAL))
WORD, NUM = re.compile(r'[\w.@-]{3,}'), re.compile(r'[\w.@-]*\d{3,}[\w.@-]*')

The wall. `ingest` builds the vault from a folder into a temp file, so a stateless container needs no volume. `private_terms` is every number in a document the vault holds private, plus the names a person named. `sealed` refuses a prompt carrying one, and refuses to hand back a failed turn as a plan. `inside` keeps the files and loses the shell, the network and delegation; `outside` gets the web and an empty folder.

In [ ]:
class Wall(Exception): pass

def ingest(folder, private=(), public=()):
    v = Vault(Path(mkdtemp())/'vault.db')
    v.add(str(folder))
    for p in private: v.mark_pii(str(p), reason='marked by a person')
    for p in public: v.mark_not_pii(str(p), reason='cleared by a person')
    return v

def private_terms(vault, names=(), ner=True):
    out = {str(n).lower() for n in names}
    for d in vault.docs:
        if vault.pii(d['source'], ner=ner).has_pii:
            out |= {x.lower() for x in NUM.findall(vault.document(d['source']).text)}
    return out

def sealed(agent, vault, names=(), ner=True):
    terms, wire = private_terms(vault, names, ner), []
    def guard(f):
        def go(prompt, **kw):
            if hit := terms & {w.lower() for w in WORD.findall(str(prompt))}: raise Wall(sorted(hit))
            if (r := pii_report(str(prompt), ner=ner)).has_pii: raise Wall(dict(r.identifying))
            wire.append(prompt)
            out = f(prompt, **kw)
            if str(out).startswith('the assistant failed'): raise Wall(out)
            return out
        return go
    agent.ask, agent.stream, agent.wire = guard(agent.ask), guard(agent.stream), wire
    return agent

def inside(vault, folder, model=LOCAL, keep=('file', 'code', 'memory', 'notebook', 'ask')):
    h = VaultHost([str(folder)], vault=vault, web=False, index=False)
    h.without |= h.provides - set(keep)
    return Agent(h, model=model, subagents=False, extensions=False)

def outside(vault, names=(), model=CLOUD):
    a = Agent(LocalHost([mkdtemp()], index=False), model=model, readonly=True,
              subagents=False, extensions=False)
    return sealed(a, vault, names)

Patterns are arithmetic; judgement is not. A person marks 8851 (a first name and a relationship) and the callback note (case numbers), and clears the sandbox card in the runbook.

In [ ]:
folder = Path(mkdtemp())/'inbox'
copytree('inbox', folder)

NAMES = ['Amara', 'Okafor', 'Tomas', 'Nowak', 'Jane']

v = ingest(folder,
           private=[folder/'letters/refund-8851.md', folder/'notes/callback-2024-03-14.md'],
           public=[folder/'ops/refund-runbook.md'])
L(v.docs).map(lambda d: (Path(d['source']).name, v.pii(d['source'], ner=True).has_pii))

A name without an honorific is exactly what the detector cannot see, so a person names it and the vault supplies every number beside it.

In [ ]:
sorted(private_terms(v, NAMES))[:14]

`refuse` returns the finding instead of an answer. `local` sends the question to the weights on the device, and to nothing else.

In [ ]:
q = 'which refunds came back from the bank, and how much is still outstanding?'
v.ask(q, pii='refuse', model=DEVICE).answer, v.ask(q, pii='local', model=DEVICE).answer

The brief is a schema rather than prose: shape and quantity, refused if the vault held it back or had to redact what the model wrote.

In [ ]:
BRIEF = 'problem:str, n_cases:int, rails:str, constraint:str, outcome:str'

def brief(vault, question, schema=BRIEF):
    a = vault.ask(question, pii='local', model=DEVICE, schema=schema, sections=6)
    if a.get('refused') or a.fields is None: raise Wall(a.answer)
    if a.get('leaked'): raise Wall(dict(a.leaked))
    return a.fields

b = brief(v, 'what is going wrong across these refund cases, in shape and quantity only?')
b

The hosted agent may only propose, and has no path to a record to propose about.

In [ ]:
ASK = """A back office refunds collections taken under direct debit mandates and cannot show you the records.
{problem} Across {n_cases} cases the money went out on {rails}, and {outcome}. {constraint}
Find the scheme rule governing the return of a collected amount, and give me a numbered runbook."""

up = outside(v, NAMES)
plan = up.ask(ASK.format(**b))
print(plan)

The plan comes back off the open web, so the agent that applies it holds files and nothing else.

In [ ]:
APPLY = """Rewrite ops/refund-runbook.md to follow this procedure. Put the cases it changes, and where
each refund should go, in ops/open-cases.md.

{plan}"""

down = inside(v, folder)
print(down.ask(APPLY.format(plan=plan)))

Every prompt that crossed, and what the half holding the records cannot do.

In [ ]:
print(up.wire[-1])
L(down.tools).attrgot('__name__').filter(lambda n: n in ('run_shell', 'read_url', 'delegate_search'))

Swap the folder, the models and the schema. `ingest`, `sealed`, `inside` and `outside` are the whole pattern.